<a href="https://colab.research.google.com/github/nastanelina143-dot/ab-test-significance-analysis/blob/main/ab_test_significance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
data = pd.read_csv('/content/drive/MyDrive/Mate_homework/ab_test.csv')

In [3]:
data.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-06,(not set),desktop,(not set),Paid Search,2,2,new account,1
1,2020-11-09,(not set),mobile,Americas,Organic Search,2,2,new account,1
2,2020-11-11,(not set),desktop,Africa,Direct,2,1,new account,1
3,2020-11-12,(not set),mobile,(not set),Organic Search,2,1,new account,1
4,2020-11-13,(not set),mobile,Africa,Organic Search,2,2,new account,1


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   value       800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB


#Data cleaning

In [5]:
data['date'] = pd.to_datetime(data['date'])

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  int64         
 6   test_group  800996 non-null  int64         
 7   event_name  800996 non-null  object        
 8   value       800996 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 55.0+ MB


In [7]:
print(f'Reviewed period: {data['date'].min()} - {data['date'].max()} ({data['date'].nunique()} days)')

Reviewed period: 2020-11-01 00:00:00 - 2021-01-27 00:00:00 (88 days)


In [8]:
data['country'].nunique()

108

In [9]:
data['country'].value_counts()

,count
country,
United States,53507
India,38123
Canada,35165
United Kingdom,26455
(not set),22063
...,...
Bahamas,601
Honduras,593
Bahrain,589


In [10]:
data = data.replace({'(not set)': 'Unknown'})

In [11]:
data['device'].value_counts()

,count
device,
desktop,419152
mobile,335832
tablet,46012


In [12]:
data['continent'].value_counts()

,count
continent,
Europe,289723
Asia,285747
Americas,171834
Africa,28652
Oceania,19040
Unknown,6000


In [13]:
data['channel'].value_counts()

,count
channel,
Organic Search,237152
Paid Search,204056
Direct,188716
Social Search,96068
Undefined,75004


In [14]:
data['event_name'].value_counts()

,count
event_name,
session,107210
session_start,106242
page_view,101907
user_engagement,94520
first_visit,81621
scroll,73643
view_promotion,61695
view_item,44869
session with orders,25892


In [15]:
data = data.replace({'new account': 'new_account'})

In [16]:
data['test'] = data['test'].astype(str)
data['test_group'] = data['test_group'].astype(str)

In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        800996 non-null  datetime64[ns]
 1   country     800996 non-null  object        
 2   device      800996 non-null  object        
 3   continent   800996 non-null  object        
 4   channel     800996 non-null  object        
 5   test        800996 non-null  object        
 6   test_group  800996 non-null  object        
 7   event_name  800996 non-null  object        
 8   value       800996 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(7)
memory usage: 55.0+ MB


In [18]:
data['test'].value_counts()

,count
test,
4,302824
3,200757
2,156791
1,140624


In [19]:
data['test_group'].value_counts()

,count
test_group,
1,400812
2,400184


In [20]:
data['value'].describe()

,value
count,800996.000000
mean,9.475863
std,37.493267
min,1.000000
25%,1.000000
50%,2.000000
75%,5.000000
max,1575.000000


The dataset has 800,996 rows and 9 columns: date, country, device, continent, channel, test, test_group, event_name and value. The data covers 88 days, from 1 November 2020 to 27 January 2021, and has four tests with two test groups each (1 and 2).

There are no empty values, but there are hidden ones. The text “(not set)” appears in the country and continent columns, so it was replaced with “Unknown”. The “Undefined” value in the channel column is kept as a separate category, because it describes an unknown traffic source and not a data error. The device column is clean and has only three values: desktop, mobile and tablet.

Data types and names also were changed. The date column was converted from text to a date format. The test and test_group columns were converted from numbers to text, because they are labels and not numeric values. In the event_name column, “new account” was renamed to “new_account”, so it looks the same as the other event names.

#Calculations

In [21]:
def ab_test_significance(data, event_name, test_number):
  test_data = data[data['test'] == test_number]

  test_data_denom = test_data[test_data['event_name'] == 'session'].groupby('test_group')['value'].sum()
  test_data_numer = test_data[test_data['event_name'] == event_name].groupby('test_group')['value'].sum()

  result = pd.merge(test_data_numer, test_data_denom, on='test_group', how='inner', suffixes=('_numer', '_denom')).reset_index()
  result['conversion'] = result['value_numer'] / result['value_denom']

  if len(result) != 2 or result.loc[0, 'conversion'] == 0:
    return None

  z_stat, p_value = sm.stats.proportions_ztest([result.loc[1, 'value_numer'], result.loc[0, 'value_numer']], [result.loc[1, 'value_denom'], result.loc[0, 'value_denom']])

  result_dict = {
      'test_number': test_number,
      'metric': f'{event_name}/session',
      'numerator_event': event_name,
      'denominator_event': 'session',
      'numerator_count_1': result.loc[0, 'value_numer'],
      'denominator_count_1': result.loc[0, 'value_denom'],
      'conversion_1': result.loc[0, 'conversion'],
      'numerator_count_2': result.loc[1, 'value_numer'],
      'denominator_count_2': result.loc[1, 'value_denom'],
      'conversion_2': result.loc[1, 'conversion'],
      'metric_change': (result.loc[1, 'conversion'] / result.loc[0, 'conversion'] - 1) * 100,
      'z_stat': z_stat,
      'p_value': p_value,
      'significant': p_value < 0.05

  }

  return result_dict

In [22]:
ab_test_significance(data, 'add_payment_info', '1')

{'test_number': '1',
 'metric': 'add_payment_info/session',
 'numerator_event': 'add_payment_info',
 'denominator_event': 'session',
 'numerator_count_1': np.int64(1988),
 'denominator_count_1': np.int64(45362),
 'conversion_1': np.float64(0.04382522816454301),
 'numerator_count_2': np.int64(2229),
 'denominator_count_2': np.int64(45193),
 'conversion_2': np.float64(0.049321797623525766),
 'metric_change': np.float64(12.542021317825736),
 'z_stat': np.float64(3.924884023129521),
 'p_value': np.float64(8.677153267271231e-05),
 'significant': np.True_}

In [23]:
results = []
events = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account']
breakdowns = ['device', 'channel', 'continent']

for test in data['test'].unique():
  for event in events:
    row = ab_test_significance(data, event, test)
    if row is not None:
      row['breakdown'] = 'total'
      row['breakdown_value'] = 'Total'
      results.append(row)

    for breakdown in breakdowns:
      for value in data[breakdown].unique():
        data_part = data[data[breakdown] == value]
        row = ab_test_significance(data_part, event, test)
        if row is not None:
          row['breakdown'] = breakdown
          row['breakdown_value'] = value
          results.append(row)

In [24]:
result_df = pd.DataFrame(results)

In [25]:
result_df.head()

,test_number,metric,numerator_event,denominator_event,numerator_count_1,denominator_count_1,conversion_1,numerator_count_2,denominator_count_2,conversion_2,metric_change,z_stat,p_value,significant,breakdown,breakdown_value
0,2,add_payment_info/session,add_payment_info,session,2344,50637,0.046290,2409,50244,0.047946,3.576911,1.240994,0.214608,False,total,Total
1,2,add_payment_info/session,add_payment_info,session,1314,29497,0.044547,1401,29380,0.047686,7.045601,1.815587,0.069434,False,device,desktop
2,2,add_payment_info/session,add_payment_info,session,978,20017,0.048858,961,19756,0.048643,-0.440088,-0.099562,0.920692,False,device,mobile
3,2,add_payment_info/session,add_payment_info,session,52,1123,0.046305,47,1108,0.042419,-8.391766,-0.445631,0.655864,False,device,tablet
4,2,add_payment_info/session,add_payment_info,session,539,13329,0.040438,571,13187,0.043300,7.077668,1.163530,0.244615,False,channel,Paid Search


In [26]:
result_df.to_csv('results.csv', index=False)

#Explanation
The code has two parts: a function that calculates the result for one metric, and a loop that runs this function for all tests and all slices of the data.

The function ab_test_significance takes a table, an event name and a test number. First, it keeps only the rows of the chosen test. Then it counts the sessions in each test group (the sum of value for the event “session”), and the number of events in each group (the sum of value for the chosen event, for example add_payment_info). Conversion is the number of events divided by the number of sessions. We calculate it for group 1 (control) and group 2 (test).

Next, the function checks the data. If one of the groups is missing, or the control conversion is zero, it returns None, because the test cannot be calculated. This often happens in small slices of data. If everything is fine, the function runs a two-proportion z-test, which compares the two conversion rates and gives a z-statistic and a p-value. If the p-value is lower than 0.05, the difference is statistically significant. The function also calculates the metric change, which shows how many percent the test group is higher or lower than the control group. The result is returned as one row with all numbers needed for the dashboard.

The loop calls this function for each of the four tests and each of the four metrics: add_payment_info, add_shipping_info, begin_checkout and new_account. First, it calculates the result for the whole test and marks it as “total”. After that, it repeats the calculation for every value of device, channel and continent. For this, it filters the data by one value (for example, “mobile”) and passes the smaller table to the same function. Rows with None are skipped. Each result gets two labels, breakdown and breakdown_value, so we know which slice it belongs to. Only the names of the four metrics are written manually. The number of tests, breakdowns and their values come from the data, so the code works if new metrics or slices are added.

At the end, all rows are collected in one table with 240 rows and saved as a CSV file. This file is used in Tableau to show the metrics and where the difference is statistically significant.

#Conclusions
The analysis of the total results shows that only the first test improved the metrics. In test 1, the conversion rate in the test group was higher at three steps of the checkout: add_payment_info (+12.5%), add_shipping_info (+6.6%) and begin_checkout (+6.7%). All three differences are statistically significant. The number of new accounts was a bit lower (-3.4%), but this change is not significant, so it may be random.

In test 2, none of the four metrics changed significantly, so the new version did not have a clear effect. In test 3, only begin_checkout changed significantly, and it went down by 3.4%. In test 4, two metrics were significantly lower in the test group: begin_checkout (-2.4%) and new_account (-3.4%). It looks like the versions in tests 3 and 4 work worse than the control version.

Based on these results, the version from test 1 looks the most promising, so the company could try it again on a bigger group of users or roll it out step by step, and watch if the improvement stays.

The versions from tests 3 and 4 seem to work worse than the control version, so it is probably better not to launch them now. The team could look at what exactly was changed in these tests and try to find the reason for the lower conversion at the beginning of the checkout. For test 2, there is no clear effect, so the team may decide if the change is worth the cost, or test a stronger idea instead.

#CSV File
https://drive.google.com/file/d/1M96nUHiBDv1-PQ6-vmHksE8jvOQlgshE/view?usp=sharing

#Tableau Dashboard
https://public.tableau.com/app/profile/nastasiia.nelina/viz/AB-testing_17935414014350/AB-testDashboard?publish=yes